# Referência Pareto 8D — NSGA-III + MOEA/D com checkpoints

Gera seis execuções independentes com direções Das–Dennis (`p=8`), sementes 1010, 2110 e 3070 e o mesmo RSM/restrição esférica da v0. Cada execução é retomável por um checkpoint serializado e também exporta a última população validada em NPZ. As sementes usadas aqui devem permanecer distintas das sementes usadas na comparação dos métodos.

In [ ]:
from pathlib import Path
import hashlib, json, os, platform, sys, time
import dill
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from pymoo.core.problem import Problem
from pymoo.core.repair import Repair
from pymoo.algorithms.moo.nsga3 import NSGA3
from pymoo.algorithms.moo.moead import MOEAD
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.util.ref_dirs import get_reference_directions
import pymoo, sklearn

def project_root(start=Path.cwd()):
    for p in (start.resolve(), *start.resolve().parents):
        if (p / 'AGENTS.md').exists() and (p / 'notebooks').exists():
            return p
    raise FileNotFoundError('Raiz do projeto não encontrada.')

ROOT = project_root()
OUT = ROOT / 'results' / 'applied' / 'reference_8d'
CKPT = OUT / 'checkpoints'
RUNS = OUT / 'runs'
for p in (OUT, CKPT, RUNS): p.mkdir(parents=True, exist_ok=True)

EXCEL_CANDIDATES = [
    ROOT / 'data' / 'VRF_artigo.xlsx',
    Path.home() / 'Documents' / 'Dissertação' / '02_REFERENCIAS' / 'Dados' / 'VRF_artigo.xlsx',
    Path.home() / 'Documents' / 'Dissertação' / '04_CODIGOS' / 'notebooks' / 'files' / 'VRF_artigo.xlsx',
]
EXCEL = next((p for p in EXCEL_CANDIDATES if p.exists()), None)
if EXCEL is None: raise FileNotFoundError('VRF_artigo.xlsx não encontrado.')

DECISION_COLS = ['cs', 'f', 'md']
OBJECTIVE_COLS = ['T', 'MTTF', 'WR', 'Ra', 'Rt', 'Kp', 'ROI', 'OEE']
OBJECTIVE_SENSE = {'T':'max','MTTF':'max','WR':'min','Ra':'min','Rt':'min','Kp':'min','ROI':'max','OEE':'max'}
SIGNS = np.array([1.0 if OBJECTIVE_SENSE[c] == 'min' else -1.0 for c in OBJECTIVE_COLS])
ALPHA = 2 ** 0.75
SEEDS = (1010, 2110, 3070)
N_PARTITIONS = 8
N_GEN = 300
CHECKPOINT_EVERY = 25
MOEAD_NEIGHBORS = 40
SBX_ETA = 20
PM_ETA = 20
SCHEMA_VERSION = 1

raw = pd.read_excel(EXCEL)
raw.columns = [str(c).strip() for c in raw.columns]
required = DECISION_COLS + OBJECTIVE_COLS
if any(c not in raw for c in required): raise ValueError('Colunas ausentes no Excel.')
data = raw[required].apply(pd.to_numeric, errors='coerce')
if data.isna().any().any(): raise ValueError('Dados ausentes ou não numéricos.')
rsm = Pipeline([('poly', PolynomialFeatures(2, include_bias=False)), ('reg', LinearRegression())])
rsm.fit(data[DECISION_COLS].to_numpy(float), data[OBJECTIVE_COLS].to_numpy(float))
poly = rsm.named_steps['poly']; reg = rsm.named_steps['reg']
B = np.vstack([reg.intercept_, reg.coef_.T])

def design(X):
    X = np.atleast_2d(np.asarray(X, float)); x1, x2, x3 = X.T
    return np.column_stack([np.ones(len(X)), x1, x2, x3, x1*x1, x1*x2, x1*x3, x2*x2, x2*x3, x3*x3])

# Confirma que a matriz rápida usa exatamente a ordem do PolynomialFeatures.
probe = np.array([[0.,0.,0.], [.2,-.4,.7]])
if not np.allclose(design(probe) @ B, rsm.predict(probe), rtol=1e-12, atol=1e-12):
    raise RuntimeError('Ordem dos termos do RSM incompatível.')

def project_sphere(X):
    X = np.asarray(X, float)
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    return X * np.minimum(1.0, ALPHA / np.maximum(norms, 1e-15))

class SphereRepair(Repair):
    def _do(self, problem, X, **kwargs): return project_sphere(X)

class RSM8DProblem(Problem):
    def __init__(self, B):
        super().__init__(n_var=3, n_obj=8, xl=-ALPHA, xu=ALPHA)
        self.B = np.asarray(B, float); self.actual_evaluations = 0
    def _evaluate(self, X, out, *args, **kwargs):
        Xp = project_sphere(X); X[:] = Xp
        self.actual_evaluations += len(Xp)
        out['F'] = (design(Xp) @ self.B) * SIGNS

ref_dirs = get_reference_directions('das-dennis', 8, n_partitions=N_PARTITIONS)
POP_SIZE = len(ref_dirs)
assert POP_SIZE == 6435
excel_hash = hashlib.sha256(EXCEL.read_bytes()).hexdigest()
config = {'schema_version':SCHEMA_VERSION, 'excel_sha256':excel_hash, 'seeds':SEEDS, 'n_partitions':N_PARTITIONS, 'pop_size':POP_SIZE, 'n_gen':N_GEN, 'checkpoint_every':CHECKPOINT_EVERY, 'moead_neighbors':MOEAD_NEIGHBORS, 'objective_sense':OBJECTIVE_SENSE, 'alpha':ALPHA, 'pymoo':pymoo.__version__, 'sklearn':sklearn.__version__}
config_hash = hashlib.sha256(json.dumps(config, sort_keys=True, default=list).encode()).hexdigest()
(OUT / 'reference_run_config.json').write_text(json.dumps({**config, 'config_hash':config_hash, 'excel':str(EXCEL), 'python':sys.version, 'platform':platform.platform()}, indent=2, default=list), encoding='utf-8')
print(json.dumps(config, indent=2, default=list))


In [ ]:
def atomic_pickle(obj, path):
    tmp = path.with_suffix(path.suffix + '.tmp')
    with tmp.open('wb') as fh: dill.dump(obj, fh, protocol=dill.HIGHEST_PROTOCOL)
    os.replace(tmp, path)

def atomic_npz(path, **arrays):
    tmp = path.with_suffix('.tmp.npz')
    np.savez_compressed(tmp, **arrays)
    os.replace(tmp, path)

def make_algorithm(method):
    repair = SphereRepair()
    crossover = SBX(prob=1.0, eta=SBX_ETA, repair=repair)
    mutation = PM(prob=1/3, eta=PM_ETA, repair=repair)
    if method == 'NSGAIII':
        return NSGA3(ref_dirs=ref_dirs, pop_size=POP_SIZE, crossover=crossover, mutation=mutation, repair=repair, eliminate_duplicates=True)
    if method == 'MOEAD':
        return MOEAD(ref_dirs=ref_dirs, n_neighbors=MOEAD_NEIGHBORS, prob_neighbor_mating=0.9, crossover=crossover, mutation=mutation, repair=repair)
    raise ValueError(method)

def generation_of(algorithm):
    return max(0, int((algorithm.n_gen or 1) - 1))

def export_population(algorithm, method, seed, final=False):
    X = project_sphere(np.asarray(algorithm.pop.get('X'), float))
    F_original = design(X) @ B
    norm = np.linalg.norm(X, axis=1)
    if np.any(norm > ALPHA + 1e-9): raise RuntimeError('População inviável.')
    suffix = 'final' if final else f'gen{generation_of(algorithm):04d}'
    path = RUNS / f'{method}_seed{seed}_{suffix}.npz'
    atomic_npz(path, X=X, F_original=F_original, method=np.array(method), seed=np.array(seed), generation=np.array(generation_of(algorithm)), config_hash=np.array(config_hash))
    return path

def run_one(method, seed):
    key = f'{method}_seed{seed}'
    state_path = CKPT / f'{key}.pkl'
    meta_path = CKPT / f'{key}.json'
    final_path = RUNS / f'{key}_final.npz'
    if final_path.exists():
        d = np.load(final_path, allow_pickle=False)
        if str(d['config_hash']) == config_hash and int(d['generation']) >= N_GEN:
            print(f'{key}: final compatível já existe; pulando.')
            return
    if state_path.exists():
        with state_path.open('rb') as fh: state = dill.load(fh)
        if state['config_hash'] != config_hash: raise RuntimeError(f'Checkpoint incompatível: {state_path}')
        algorithm = state['algorithm']; elapsed_before = state['elapsed_seconds']
        print(f'{key}: retomando após geração {generation_of(algorithm)}.')
    else:
        problem = RSM8DProblem(B)
        algorithm = make_algorithm(method)
        algorithm.setup(problem, termination=('n_gen', N_GEN), seed=int(seed), verbose=False)
        elapsed_before = 0.0
        print(f'{key}: iniciando.')
    last_checkpointed_gen = generation_of(algorithm) if state_path.exists() else -1
    started = time.perf_counter()
    while algorithm.has_next():
        algorithm.next()
        gen = generation_of(algorithm)
        # MOEA/D é loop-wise: next() também é chamado dentro de uma geração.
        # Só serializar na transição de geração, quando o gerador interno está vazio.
        should_checkpoint = (gen != last_checkpointed_gen) and (gen % CHECKPOINT_EVERY == 0 or not algorithm.has_next())
        if should_checkpoint:
            elapsed = elapsed_before + time.perf_counter() - started
            snap = export_population(algorithm, method, seed, final=False)
            atomic_pickle({'config_hash':config_hash, 'algorithm':algorithm, 'elapsed_seconds':elapsed}, state_path)
            meta = {'method':method, 'seed':seed, 'generation':gen, 'target_generations':N_GEN, 'elapsed_seconds':elapsed, 'evaluations':algorithm.problem.actual_evaluations, 'snapshot':str(snap), 'config_hash':config_hash}
            tmp = meta_path.with_suffix('.json.tmp'); tmp.write_text(json.dumps(meta, indent=2), encoding='utf-8'); os.replace(tmp, meta_path)
            last_checkpointed_gen = gen
            print(f'{key}: checkpoint geração {gen}/{N_GEN}; {elapsed/60:.1f} min.')
    export_population(algorithm, method, seed, final=True)
    print(f'{key}: concluído.')

print(f'{POP_SIZE:,} indivíduos × {N_GEN} gerações × 6 execuções = {POP_SIZE*N_GEN*6:,} avaliações nominais.')


In [ ]:
# Execução sequencial para evitar paralelismo aninhado e pressão de RAM.
for method in ('NSGAIII', 'MOEAD'):
    for seed in SEEDS:
        run_one(method, seed)
print('Todas as execuções de referência foram concluídas.')
